In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

In [2]:
spark = (
        SparkSession.builder
        .appName("Joins App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    );

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dfd7f6f0-27d4-4994-b776-9f2a445e3540;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [3]:
sc = spark.sparkContext

In [4]:
sc

<SparkContext master=local[*] appName=Adaptive Query Execution App>

In [5]:
spark

26/04/28 21:55:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
emp_data = [
    (1, "John", 101),
    (2, "Mike", 102),
    (3, "Sara", 103),
    (4, "Nina", None),
    (5, "Tom", None),
    (6, "Emma", 101),
    (7, "Chris", 102),
    (8, "Sophia", 104),
    (9, "David", None),
    (10, "Liam", 102)]

In [7]:
dept_data = [(101, "HR"), (102, "Finance"), (103, "IT"), (104, "Marketing")]

In [8]:
employees = spark.createDataFrame(emp_data, ["emp_id", "name", "dept_id"])

In [9]:
departments = spark.createDataFrame(dept_data, ["dept_id", "dept_name"])

### Inner Join

In [10]:
employees.join(departments, "dept_id", "inner").show()

+-------+------+------+---------+
|dept_id|emp_id|  name|dept_name|
+-------+------+------+---------+
|    101|     1|  John|       HR|
|    101|     6|  Emma|       HR|
|    102|     2|  Mike|  Finance|
|    102|     7| Chris|  Finance|
|    102|    10|  Liam|  Finance|
|    103|     3|  Sara|       IT|
|    104|     8|Sophia|Marketing|
+-------+------+------+---------+



### Left Join

In [11]:
employees.join(departments, "dept_id", "left").show()

+-------+------+------+---------+
|dept_id|emp_id|  name|dept_name|
+-------+------+------+---------+
|    101|     1|  John|       HR|
|    102|     2|  Mike|  Finance|
|    103|     3|  Sara|       IT|
|   NULL|     4|  Nina|     NULL|
|   NULL|     5|   Tom|     NULL|
|    101|     6|  Emma|       HR|
|    102|     7| Chris|  Finance|
|    104|     8|Sophia|Marketing|
|   NULL|     9| David|     NULL|
|    102|    10|  Liam|  Finance|
+-------+------+------+---------+



### Left Semi Join

In [12]:
employees.join(departments, "dept_id", "left_semi").show()

[Stage 12:>                                                       (0 + 12) / 12]

+-------+------+------+
|dept_id|emp_id|  name|
+-------+------+------+
|    101|     1|  John|
|    101|     6|  Emma|
|    102|     2|  Mike|
|    102|     7| Chris|
|    102|    10|  Liam|
|    103|     3|  Sara|
|    104|     8|Sophia|
+-------+------+------+



### Left Anti Join

In [13]:
employees.join(departments, "dept_id", "left_anti").show()

[Stage 17:===================>                                     (4 + 8) / 12]

+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|   NULL|     4| Nina|
|   NULL|     5|  Tom|
|   NULL|     9|David|
+-------+------+-----+



### Right Join

In [14]:
employees.join(departments, "dept_id", "right").show()

+-------+------+------+---------+
|dept_id|emp_id|  name|dept_name|
+-------+------+------+---------+
|    101|     6|  Emma|       HR|
|    101|     1|  John|       HR|
|    102|    10|  Liam|  Finance|
|    102|     7| Chris|  Finance|
|    102|     2|  Mike|  Finance|
|    103|     3|  Sara|       IT|
|    104|     8|Sophia|Marketing|
+-------+------+------+---------+



In [15]:
spark.stop()